# End to end transformer fine-tuning

## Dataset preparation

In [2]:
from datasets import load_dataset
dataset = load_dataset("Edoh/manim_python")

Generating test split: 100%|██████████| 51/51 [00:00<00:00, 19487.06 examples/s]


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 599
    })
    test: Dataset({
        features: ['instruction', 'output'],
        num_rows: 51
    })
})

In [4]:
dataset['train'][0]

{'instruction': "Create a new scene named 'MyScene'.",
 'output': 'from manim import * class MyScene(Scene): def construct(self): pass'}

In [6]:
# Load tokenizer
from transformers import GPT2Tokenizer
model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [7]:
def preprocess_data(example):
    inputs = [
        f'instruction": {instr}\n Output: {out}' for instr, out in zip(example["instruction"], example["output"])
    ]
    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

## Tại sao cần tokenized["labels"] = tokenized["input_ids"].copy()?

  ### Bối cảnh: Đang fine-tune GPT-2

  Notebook này đang fine-tune model GPT-2 — một Causal Language Model (mô hình ngôn ngữ tự hồi quy). Nhiệm vụ huấn luyện của GPT-2 là: dự đoán token tiếp theo
  dựa trên các token trước đó.

  ### input_ids là gì?

  Khi bạn gọi tokenizer(inputs, ...), kết quả trả về là một dictionary chứa:

  • input_ids: danh sách các token ID (số nguyên) đại diện cho văn bản đầu vào.
  • attention_mask: mask cho biết token nào là thật, token nào là padding.

  ### labels là gì và tại sao lại bằng input_ids?

  Thư viện Hugging Face Transformers yêu cầu trường labels để tính loss (hàm mất mát) trong quá trình huấn luyện:

  1. Không có labels → model chỉ chạy forward pass, không tính loss, không học được gì.
  2. Có labels → model sẽ tính Cross-Entropy Loss giữa dự đoán và labels.
  3. labels = input_ids vì với Causal LM, mục tiêu huấn luyện là:
  │ Cho chuỗi token [A, B, C, D], model cần học:
  │
  │     • Từ A → dự đoán B
  │     • Từ A, B → dự đoán C
  │     • Từ A, B, C → dự đoán D
  Nói cách khác, đầu vào và đầu ra (target) là cùng một chuỗi, chỉ lệch nhau 1 vị trí. Hugging Face tự động xử lý việc dịch (shift) sang phải 1 vị trí bên trong
  model, nên bạn chỉ cần gán labels = input_ids.

  ### Tại sao dùng .copy()?

  Dùng .copy() để tạo một bản sao độc lập. Nếu không copy, labels và input_ids sẽ trỏ đến cùng một vùng nhớ — khi thay đổi một cái (ví dụ, mask padding tokens
  trong labels thành -100), cái còn lại cũng bị ảnh hưởng.

  ### Tóm tắt luồng hoạt động

    Input text: "instruction: Create a scene\n Output: from manim import ..."
                        ↓
                tokenizer(...)
                        ↓
            input_ids = [15, 42, 88, 103, ...]   ← token hóa văn bản
            labels    = [15, 42, 88, 103, ...]   ← bản sao, dùng làm target
                        ↓
                Trong model GPT-2:
                - Input:  [15, 42, 88, 103]  → Model dự đoán token tiếp theo
                - Target: [42, 88, 103, ...]  → (tự động shift bên trong)
                - Loss = CrossEntropy(dự đoán, target)

  │ Kết luận: labels chính là đáp án mà model cần học để dự đoán. Với Causal LM như GPT-2, đáp án chính là chuỗi token đầu vào (dịch phải 1 vị trí), nên labels =
  │ input_ids.copy().

In [8]:
tokenized_datasets = dataset.map(preprocess_data, batched=True, remove_columns=dataset["train"].column_names)

Map: 100%|██████████| 51/51 [00:00<00:00, 5134.65 examples/s]


In [10]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 599
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 51
    })
})

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 599
    })
    test: Dataset({
        features: ['instruction', 'output'],
        num_rows: 51
    })
})